In [3]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# NLP  (para nuvem de palavras, análise de sentimentos, etc.)
import spacy
from wordcloud import WordCloud

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

# Deep Learning (não esquecer: pip install tensorflow)
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
print(tf.__version__) # Verifica a versão instalada

# Configurações visuais
plt.style.use("ggplot")

I0000 00:00:1774796352.657388    1251 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774796353.451145    1251 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774796355.620063    1251 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


2.21.0


In [31]:
# Verifica se transformers, datasets e torch estão instalados no ambiente corretamente.
import sys
print("Python em uso:", sys.executable)

packages = ["transformers", "datasets", "torch"]

for pkg in packages:
    try:
        mod = __import__(pkg)
        print(f"{pkg} OK - versão:", getattr(mod, "__version__", "sem versão"))
    except ImportError:
        print(f"{pkg} NÃO INSTALADO")

Python em uso: /home/paulo/projeto_tech_05/venv/bin/python
transformers OK - versão: 5.3.0
datasets OK - versão: 4.8.3
torch OK - versão: 2.10.0+cu128


In [ ]:
# %pip install -q transformers datasets torch <<< n'ao precisa pois já estão instalados,
# mas é uma boa prática para garantir que o ambiente esteja configurado corretamente.
# ####### resultado da celula acima ##########################
# Python em uso: /home/paulo/projeto_tech_05/venv/bin/python
# transformers OK - versão: 5.3.0
# datasets OK - versão: 4.8.3
# torch OK - versão: 2.10.0+cu128

In [32]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


## Arquivo baixado do Kaggle

In [33]:
df = pd.read_csv("/home/paulo/Evolutions_Skills/TECH_CHALLENGE/TECH_CHALLENGE_fase_05/complaints_processed.csv")
## Remover a coluna de índice
df = df.drop(columns=["Unnamed: 0"])
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 162421 entries, 0 to 162420
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   product    162421 non-null  str  
 1   narrative  162411 non-null  str  
dtypes: str(2)
memory usage: 96.1 MB


,product,narrative
0,credit_card,purchase order day shipping amount receive pro...
1,credit_card,forwarded message date tue subject please inve...
2,retail_banking,forwarded message cc sent friday pdt subject f...
3,credit_reporting,payment history missing credit report speciali...
4,credit_reporting,payment history missing credit report made mis...


In [34]:
df.shape
df.tail()
df["product"].value_counts().head(10)

product
credit_reporting       91179
debt_collection        23150
mortgages_and_loans    18990
credit_card            15566
retail_banking         13536
Name: count, dtype: int64

In [36]:
## Calcular total por produto
total_produtos = df["product"].value_counts(normalize=False)
#total_produtos
## Calcular proporção por produto
proporcoes = df["product"].value_counts(normalize=True)
#proporcoes

print("Total de reclamações por produto:")
print(total_produtos)
print("\nProporção de reclamações por produto:")
print(proporcoes)

Total de reclamações por produto:
product
credit_reporting       91179
debt_collection        23150
mortgages_and_loans    18990
credit_card            15566
retail_banking         13536
Name: count, dtype: int64

Proporção de reclamações por produto:
product
credit_reporting       0.561374
debt_collection        0.142531
mortgages_and_loans    0.116918
credit_card            0.095837
retail_banking         0.083339
Name: proportion, dtype: float64


In [37]:
# # O número de registros por categoria foi definido de forma proporcional à
# # distribuição original dos produtos, multiplicando-se as proporções observadas
# # pelo tamanho desejado da amostra.
# total_desejado = 50000

# amostra_por_produto = (proporcoes * total_desejado).astype(int)
# amostra_por_produto

In [38]:
# # Gerar a amostra estratificada
# df = (
#     df.groupby("product", group_keys=False)
#       .apply(lambda x: x.sample(
#           n=min(len(x), amostra_por_produto[x.name]),
#           random_state=42
#       ))
# )

In [39]:
## >>> DECIDIR <<<
## Criar amostra aleatória para otimização
#df_sample = df.sample(frac=1.0, random_state=42)
# ou utiliza toda a base de 162 mil registros.
df_total = df.copy()

In [ ]:
## >>> DECIDIR <<<
## copiar o dataframe original para aplicar a limpeza e lematização em toda a base de dados com 162 mil registros.
## Qual dataframe usar daqui em diante.
# df = df.copy()  # => Rodar no noteboook colab.
df = df_total.copy()   # => Rodar na maquina local com GPU.

In [41]:
shape = df.shape
columns = df.columns

print(f"Shape do DataFrame: {shape}")
print(f"Colunas do DataFrame: {columns}")   

Shape do DataFrame: (162421, 2)
Colunas do DataFrame: Index(['product', 'narrative'], dtype='str')


In [42]:
## Limpeza básica do texto
import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

# declarando a função de limpeza de texto
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join([w for w in text.split() if w not in stop_words])
    return text

[nltk_data] Downloading package stopwords to /home/paulo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [43]:
# inserido para lematizar (rais das) palavras.
from nltk.stem import LancasterStemmer
stemmer = LancasterStemmer()

In [44]:
# função que aplica a lematização.
def stem_text(text):
    return ' '.join([stemmer.stem(w) for w in text.split()])

In [45]:


#  stem_text para lematizar as palavras base de dados com todos os 162 mil registros.
df["stem_narrative"] = (
    df["narrative"]
    .fillna("")
    .astype(str)
    .apply(clean_text)
    .apply(stem_text)
)

In [49]:
columns = df.columns
print(f"Colunas do DataFrame: {columns}")
head = df.head()
print(head)
tail = df.tail()
print(tail)

Colunas do DataFrame: Index(['product', 'narrative', 'stem_narrative'], dtype='str')
            product                                          narrative  \
0       credit_card  purchase order day shipping amount receive pro...   
1       credit_card  forwarded message date tue subject please inve...   
2    retail_banking  forwarded message cc sent friday pdt subject f...   
3  credit_reporting  payment history missing credit report speciali...   
4  credit_reporting  payment history missing credit report made mis...   

                                      stem_narrative  
0  purchas ord day ship amount receiv produc week...  
1  forward mess dat tue subject pleas investig co...  
2  forward mess cc sent friday pdt subject fin le...  
3  pay hist miss credit report spec loan serv sl ...  
4  pay hist miss credit report mad mistak put acc...  
                 product narrative stem_narrative
162416   debt_collection      name            nam
162417       credit_card      name      

In [50]:
# Verificando se o ambiente de deep learning está configurado corretamente.
import torch
torch.cuda.is_available()
print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


In [51]:
# %pip install -q transformers datasets torch
# Verifica bibliotecas Novamente, para garantir que o ambiente esteja configurado corretamente.
import sys
print("Python em uso:", sys.executable)

packages = ["transformers", "datasets", "torch"]

for pkg in packages:
    try:
        mod = __import__(pkg)
        print(f"{pkg} OK - versão:", getattr(mod, "__version__", "sem versão"))
    except ImportError:
        print(f"{pkg} NÃO INSTALADO")

Python em uso: /home/paulo/projeto_tech_05/venv/bin/python
transformers OK - versão: 5.3.0
datasets OK - versão: 4.8.3
torch OK - versão: 2.10.0+cu128


In [52]:
## Importar bibliotecas
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import Dataset

In [53]:
import torch
import gc

# 1. Remova as variáveis pesadas (substitua pelos nomes dos seus objetos)
#del model
#del tokenizer
# del model

# 2. Limpe o lixo da memória RAM
gc.collect()

# 3. Esvazie o cache da GPU para o nvidia-smi mostrar como livre
torch.cuda.empty_cache()

In [54]:
## Garantir string limpa
df["stem_narrative"] = (
    df["stem_narrative"]
    .fillna("")
    .astype(str)
)

In [55]:
## Garantir que todos os textos são strings
texts = df["stem_narrative"].fillna("").astype(str).tolist()
type(texts)
type(texts[160000]) # caso use toda base.

str

In [57]:
# verificar o comprimento dos textos para garantir que estão dentro do limite do
# modelo (512 tokens para BERT).
lengths = [len(t.split()) for t in texts]
print(f"Comprimento médio dos textos: {np.mean(lengths):.2f}")
print(f"Comprimento máximo dos textos: {np.max(lengths)}")
print(f"Mediana do comprimento dos textos: {np.median(lengths):.2f}")
print(f"Desvio padrão do comprimento dos textos: {np.std(lengths):.2f}")
print(f"Variancia do comprimento dos textos: {np.var(lengths):.2f}")
print(f"Quantidade de textos com mais de 512 tokens: {np.sum([l > 512 for l in lengths])}")
print(f"Porcentagem de textos com mais de 512 tokens: {np.mean([l > 512 for l in lengths]) * 100:.2f}%")
print(f"Quantidade de textos com 512 tokens ou menos: {np.sum([l <= 512 for l in lengths])}")

Comprimento médio dos textos: 53.05
Comprimento máximo dos textos: 139
Mediana do comprimento dos textos: 50.00
Desvio padrão do comprimento dos textos: 29.91
Variancia do comprimento dos textos: 894.80
Quantidade de textos com mais de 512 tokens: 0
Porcentagem de textos com mais de 512 tokens: 0.00%
Quantidade de textos com 512 tokens ou menos: 162421


In [56]:
# Para garantir que todos os textos estejam dentro do limite de 512 tokens,
# vamos truncar os textos mais longos.
#
# Como o modelo BERT tem um limite de 512 tokens, vamos truncar os textos mais
# longos para garantir que todos estejam dentro desse limite.
texts = [t[:512] for t in texts]
print(len(texts))

162421


In [58]:
# ################################### #
## CARREGAR MODELO >>>> FinancialBERT #
# ################################### #
# carregar o pipeline de análise de sentimentos usando um modelo pré-treinado
# específico para análises financeiras, garantindo que ele utilize a GPU para
# acelerar o processamento.
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0  # use GPU device 0; set -1 to force CPU
)
# ################################### #
## CARREGAR MODELO >>>> FinancialBERT #
# ################################### #

# model_name = "ahmedrachid/FinancialBERT-Sentiment-Analysis"

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(model_name)

# model = AutoModelForSequenceClassification.from_pretrained(model_name)
# model = model.to('cuda')  # Move model to GPU

# # garantir que o pipeline também use a GPU
# sentiment_pipeline = pipeline(
#     "sentiment-analysis",
#     model=model,
#     tokenizer=tokenizer,
#     device=0  # Specify GPU device (0 = GPU device, -1 = CPU)
# )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [59]:
# verificar se a GPU está sendo utilizada corretamente,
# imprimindo informações sobre a disponibilidade da GPU e o número de dispositivos disponíveis.
# Caso device=0 esteja configurado, o modelo deve usar a GPU.
# Se a GPU não estiver disponível, ele irá automaticamente usar a CPU,

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())

CUDA available: True
GPU device count: 1
Current device: 0


In [60]:
## Rodar classificação otimizada na GPU
results = sentiment_pipeline(
    texts,
    batch_size=256,
    truncation=True
)

In [61]:
df["sentiment_value"] = [r["label"] for r in results]
df["score"] = [r["score"] for r in results]
df.head()

,product,narrative,stem_narrative,sentiment_value,score
0,credit_card,purchase order day shipping amount receive pro...,purchas ord day ship amount receiv produc week...,neutral,0.550079
1,credit_card,forwarded message date tue subject please inve...,forward mess dat tue subject pleas investig co...,negative,0.494491
2,retail_banking,forwarded message cc sent friday pdt subject f...,forward mess cc sent friday pdt subject fin le...,neutral,0.717251
3,credit_reporting,payment history missing credit report speciali...,pay hist miss credit report spec loan serv sl ...,neutral,0.553077
4,credit_reporting,payment history missing credit report made mis...,pay hist miss credit report mad mistak put acc...,neutral,0.578120


In [62]:
## Converter para Dataset (otimização real)
dataset = Dataset.from_pandas(
    df[["sentiment_value"]].reset_index(drop=True)
)
type(dataset)
# ver os primeiros elementos do dataset
dataset[:5]

{'sentiment_value': ['neutral', 'negative', 'neutral', 'neutral', 'neutral']}

In [63]:
# Salvar como planilha com avaliação atribuida.
df.to_csv("/home/paulo/projeto_tech_05/complaints_sentiment_RoBERTa.csv", index=False)

In [10]:
# rodar depois que perdeu a conexão.
# CARREGAR A PLANILHA COM AS CLASSIFICAÇÕES DE SENTIMENTO ATRIBUÍDAS

df_Roberta = pd.read_csv("/home/paulo/projeto_tech_05/complaints_sentiment_RoBERTa.csv")
df_Roberta.head()

,product,narrative,stem_narrative,sentiment_value,score
0,credit_card,purchase order day shipping amount receive pro...,purchas ord day ship amount receiv produc week...,neutral,0.550079
1,credit_card,forwarded message date tue subject please inve...,forward mess dat tue subject pleas investig co...,negative,0.494491
2,retail_banking,forwarded message cc sent friday pdt subject f...,forward mess cc sent friday pdt subject fin le...,neutral,0.717251
3,credit_reporting,payment history missing credit report speciali...,pay hist miss credit report spec loan serv sl ...,neutral,0.553077
4,credit_reporting,payment history missing credit report made mis...,pay hist miss credit report mad mistak put acc...,neutral,0.578120


In [12]:
## Distribuição geral de sentimentos
sentimentos = df_Roberta["sentiment_value"].value_counts(normalize=False)
proporcoes = df_Roberta["sentiment_value"].value_counts(normalize=True)
print("Distribuição de sentimentos (contagem):")
print(sentimentos)
print("\nDistribuição de sentimentos (proporção):")
print(proporcoes)

Distribuição de sentimentos (contagem):
sentiment_value
neutral     118019
negative     44237
positive       165
Name: count, dtype: int64

Distribuição de sentimentos (proporção):
sentiment_value
neutral     0.726624
negative    0.272360
positive    0.001016
Name: proportion, dtype: float64


In [13]:
# Estatísticas de confiança dos scores de sentimento por categoria
print("============= MEDIA =====================")
print(df_Roberta.groupby("sentiment_value")["score"].mean())
print("")
print("============= MEDIANA =====================")
print(df_Roberta.groupby("sentiment_value")["score"].median())
print("")
print("============= MODA =====================")
print(df_Roberta.groupby("sentiment_value")["score"].apply(lambda x: x.mode()))  # modo com apply
print("")
print("============= DESVIO PADRÃO =====================")
print(df_Roberta.groupby("sentiment_value")["score"].std())
print("")
print("============= VARIÂNCIA =====================")
print(df_Roberta.groupby("sentiment_value")["score"].var())
print("")
print("============= MÁXIMO =====================")
print(df_Roberta.groupby("sentiment_value")["score"].max())
print("")
print("============= MÍNIMO =====================")
print(df_Roberta.groupby("sentiment_value")["score"].min())

============= MEDIA =====================
sentiment_value
negative    0.639450
neutral     0.735282
positive    0.606713
Name: score, dtype: float64

============= MEDIANA =====================
sentiment_value
negative    0.627488
neutral     0.759910
positive    0.573068
Name: score, dtype: float64

============= MODA =====================
sentiment_value   
negative         0    0.741462
neutral          0    0.628421
positive         0    0.401299
Name: score, dtype: float64

============= DESVIO PADRÃO =====================
sentiment_value
negative    0.101385
neutral     0.118947
positive    0.120184
Name: score, dtype: float64

============= VARIÂNCIA =====================
sentiment_value
negative    0.010279
neutral     0.014148
positive    0.014444
Name: score, dtype: float64

============= MÁXIMO =====================
sentiment_value
negative    0.938883
neutral     0.947159
positive    0.927214
Name: score, dtype: float64

============= MÍNIMO =====================
sentiment_

## MODELO ## Carregar modelo FinancialBERT

In [14]:
# De novo, verificando se o ambiente de deep learning está configurado corretamente.
import torch
torch.cuda.is_available()
print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


In [15]:
# %pip install -q transformers datasets torch
# verifica bibliotecas Novamente, para garantir que o ambiente esteja configurado corretamente.
import sys
print("Python em uso:", sys.executable)

packages = ["transformers", "datasets", "torch"]

for pkg in packages:
    try:
        mod = __import__(pkg)
        print(f"{pkg} OK - versão:", getattr(mod, "__version__", "sem versão"))
    except ImportError:
        print(f"{pkg} NÃO INSTALADO")

Python em uso: /home/paulo/projeto_tech_05/venv/bin/python
transformers OK - versão: 5.3.0
datasets OK - versão: 4.8.3
torch OK - versão: 2.10.0+cu128


In [16]:
# %pip install -q transformers datasets torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import Dataset

## Carregar modelo FinancialBERT

In [17]:
model_name = "ahmedrachid/FinancialBERT-Sentiment-Analysis"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name)
model = model.to('cuda')  # Move model to GPU

# garantir que o pipeline também use a GPU
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=0  # Specify GPU device (0 = GPU device, -1 = CPU)
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ahmedrachid/FinancialBERT-Sentiment-Analysis
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ahmedrachid/FinancialBERT-Sentiment-Analysis
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
# Copiando dataframe original para um novo dataframe que será usado para análises e visualizações, 
# garantindo que as alterações feitas no novo dataframe não afetem o dataframe original.   
df_fin_Bert = df_Roberta.copy()
df_fin_Bert.info()
df_fin_Bert.head()

<class 'pandas.DataFrame'>
RangeIndex: 162421 entries, 0 to 162420
Data columns (total 5 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   product          162421 non-null  str    
 1   narrative        162411 non-null  str    
 2   stem_narrative   162411 non-null  str    
 3   sentiment_value  162421 non-null  str    
 4   score            162421 non-null  float64
dtypes: float64(1), str(4)
memory usage: 171.6 MB


,product,narrative,stem_narrative,sentiment_value,score
0,credit_card,purchase order day shipping amount receive pro...,purchas ord day ship amount receiv produc week...,neutral,0.550079
1,credit_card,forwarded message date tue subject please inve...,forward mess dat tue subject pleas investig co...,negative,0.494491
2,retail_banking,forwarded message cc sent friday pdt subject f...,forward mess cc sent friday pdt subject fin le...,neutral,0.717251
3,credit_reporting,payment history missing credit report speciali...,pay hist miss credit report spec loan serv sl ...,neutral,0.553077
4,credit_reporting,payment history missing credit report made mis...,pay hist miss credit report mad mistak put acc...,neutral,0.578120


In [21]:
texts = df_fin_Bert["stem_narrative"].fillna("").astype(str).tolist()
type(texts)
type(texts[123456])

str

In [22]:
# verificar o comprimento dos textos para garantir que estão dentro do limite do modelo (512 tokens para BERT).
lengths = [len(t.split()) for t in texts]
print(f"Comprimento médio dos textos: {np.mean(lengths):.2f}")
print(f"Comprimento máximo dos textos: {np.max(lengths)}")
print(f"Mediana do comprimento dos textos: {np.median(lengths):.2f}")
print(f"Desvio padrão do comprimento dos textos: {np.std(lengths):.2f}")
print(f"Variancia do comprimento dos textos: {np.var(lengths):.2f}")
print(f"Quantidade de textos com mais de 512 tokens: {np.sum([l > 512 for l in lengths])}") 
print(f"Porcentagem de textos com mais de 512 tokens: {np.mean([l > 512 for l in lengths]) * 100:.2f}%")
print(f"Quantidade de textos com 512 tokens ou menos: {np.sum([l <= 512 for l in lengths])}")

Comprimento médio dos textos: 80.20
Comprimento máximo dos textos: 2684
Mediana do comprimento dos textos: 50.00
Desvio padrão do comprimento dos textos: 108.82
Variancia do comprimento dos textos: 11841.66
Quantidade de textos com mais de 512 tokens: 1696
Porcentagem de textos com mais de 512 tokens: 1.04%
Quantidade de textos com 512 tokens ou menos: 160725


## Rodar Financial Bert para comparar com RoBERTa

In [ ]:
results = sentiment_pipeline(
    texts,
    batch_size=256,
    truncation=True,
    device=0  # Specify GPU device (0 = GPU device, -1 = CPU)
)

In [ ]:
## Salvar os resultados
df_fin_Bert["sentiment_value_FinBert"] = [r["label"] for r in results]
df_fin_Bert["score_FinBert"] = [r["score_FinBert"] for r in results]

In [ ]:
# Salvar como planilha 
df_fin_Bert.to_csv("/home/paulo/projeto_tech_05/complaints_sentiment_Comparados.csv", index=False)
df_fin_Bert.head()

In [ ]:
# Caso precise,rodar depois que perdeu a conexão.
# CARREGAR o dataframe como PLANILHA

df_compados = pd.read_csv("/home/paulo/projeto_tech_05/complaints_sentiment_Comparados.csv")
df_compados.head()